# 1. Import Libraries

In [1]:
import pandas as pd
import re
from pathlib import Path

# 2. Merge Metadata Files

In [2]:
METADATA_FOLDER = Path.cwd() / 'metadata_list'

metadata_df_list = []

for path_to_file in METADATA_FOLDER.glob('*.csv'):

    print(path_to_file)
    
    sub_metadata_df = pd.read_csv(path_to_file)

    metadata_df_list.append(sub_metadata_df)

metadata_df = pd.concat(metadata_df_list, ignore_index=True)

C:\Users\young\Jupyter Notebook\metadata_list\fcc_news_release_metadata_page_0-149.csv
C:\Users\young\Jupyter Notebook\metadata_list\fcc_news_release_metadata_page_150-249.csv
C:\Users\young\Jupyter Notebook\metadata_list\fcc_news_release_metadata_page_250-331.csv


In [3]:
for col in ['released_on', 'issued_on', 'adopted']:
    metadata_df[col] = pd.to_datetime(metadata_df[col], errors='coerce')

metadata_df = metadata_df.sort_values(
    'released_on', 
    ascending=False, 
    ignore_index=True
)

In [4]:
metadata_df['download_status'].value_counts()

download_status
downloaded                 3200
skipped_no_matching_txt     111
Name: count, dtype: int64

In [5]:
metadata_df = metadata_df[metadata_df['download_status'] == 'downloaded']

In [6]:
metadata_df['downloaded_filename'] = metadata_df['downloaded_files'].apply(lambda x: Path(str(x)).name)

metadata_df['downloaded_filename_key'] = metadata_df['downloaded_filename'].str.lower()

mask_duplicates = metadata_df['downloaded_filename_key'].duplicated(keep='first')

metadata_df = metadata_df[~mask_duplicates]

In [7]:
metadata_df.to_csv('fcc_news_release_metadata.csv')

# 3. Text Body Extraction

## 3.1. Regular Expressions for News Release Formats

In [8]:
CITY_PREFIX = r'''
    (?:
        \(?\s*
        (?:
            WASHINGTON(?:\s*,?\s*D\.?\s*C\.?)?|
            BATON\s*ROUGE,\s*La\.|BARCELONA,\s*Spain|BOSTON|BRUSSELS,\s*BELGIUM|
            BUCHAREST,\s*ROMANIA|CHICAGO|HOUSTON|LITTLE\s*ROCK(?:,\s*(?:Ark\.|ARKANSAS))?|
            LAS\s*VEGAS|LOS\s*ANGELES|MIAMI|NEW\s*DELHI,\s*INDIA|OKLAHOMA\s*CITY|
            PHOENIX|PLUMAS\s*COUNTY,\s*CALIFORNIA|SEATTLE|SIOUX\s*FALLS,\s*SOUTH\s*DAKOTA|
            TAIPEI\s*CITY,\s*TAIWAN|TAMPA|TEXAS
        )
        \s*\)?
    )
'''

DATELINE = r'''
    [A-Z]{3,9}\.?,?\s+\d{1,2}(?:st|nd|rd|th)?,\s+\d{4}
'''

DASH = r'[-\u2013\u2014]'

In [9]:
DATELINE_LINE_RE = re.compile(
    rf'''(?ix)
    ^\s*
    (?:{DASH}{{1,2}}|:)?\s*
    (?:
        (?:{CITY_PREFIX})\s*,?\s*
        {DATELINE}
        (?:\s*(?:{DASH}{{1,2}}|:)\s*)?
      |
        {DATELINE}
        \s*(?:{DASH}{{1,2}}|:)\s*
        (?=[A-Z"\u201c])
    )
    '''
)

In [10]:
CITY_BODY_PREFIX_RE = re.compile(
    rf'''(?ix)
    ^\s*
    (?:{CITY_PREFIX})
    (?:\s*,\s*[A-Z][A-Za-z]+(?:\s+[A-Z][A-Za-z]+)*,?\s*[A-Z]{{2}})?
    \s*,?\s*
    (?:{DASH}{{1,2}}|:)?\s*
    (?=(?:["\u201c\u201d]?\s*)?[A-Z])
    '''
)
# \u201c: left double quotation mark(“)
# \u201d: right double quotation mark(”)

In [11]:
DATE_LINE_RE = re.compile(
    rf'''(?ix)
    ^
    (?:
        (?:FOR\s+IMMEDIATE\s+RELEASE):?\s*
    )?
    (?:Date:?\s*)?
    {DATELINE}\b
    '''
)

In [12]:
HEADER_RE = re.compile(
    r'''(?ix)
    ^(?:
        NEWS|
        ADVISORY|
        UPDATE|
        PUBLIC\s+NOTICE|
        Press\s+Office|
        U\.S\.\s+Department\s+of\s+Homeland\s+Security|
        500\s+C\s+Street|
        www\.fema\.gov|
        No\.:\s*|
        Federal\s+Communications\s+Commission\s*$|
        Nathan\s+A\.\s+Simington$|
        Commissioner$|
        445\s+12(?:th)?\s+Street|
        45\s+L\s+Street|
        th$|
        Street,\s*S\.W\.|
        Washington,?\s*D\.?\s*C\.?\s*20554|
        This\s+is\s+an\s+unofficial\s+announcement|
        constitutes\s+official\s+action|
        See\s+MCI\s+v\.\s*FCC|
        (?:\d+\s*)?\(?D\.?\s*C\.?\s*Circ|
        News\s+Media\s+Information|
        Internet:|
        TTY:|
        Fax-On-Demand|
        ftp\.fcc\.gov|
        FOR\s+IMMEDIATE\s+RELEASE|
        NEWS\s+(?:MEDIA\s+)?CONTACTS?:?|
        MEDIA\s+CONTACTS?:?|
        FCC\s+Media\s+Contacts?:?|
        FEMA\s+Media\s+Contacts?:?
    )
    '''
)

In [13]:
CONTACT_RE = re.compile(
    r'''(?ix)
    (
        \bmedia\s+contacts?\b|
        \bnews\s+contacts?\b|
        \bcontact\b|
        ^contacts?\b|
        \bemail\b|
        \be-mail\b|
        @fcc\.gov|
        fema-news-desk@|
        \(?202\)?[- .]?418|
        \(?202\)?[- .]?646|
        888[- .]?835
    )
    '''
)

In [14]:
TITLE_SKIP_RE = re.compile(
    r'''(?ix)
    ^(?:
        [-\u2013\u2014]+|
        \(?[A-Z]{1,4}\s+DOCKET\s+NO\.?|
        Re\s*:|
        STATEMENT\s+OF$
    )
    '''
)

In [15]:
LETTER_FRONT_MATTER_RE = re.compile(
    r'''(?ix)
    ^(?:
        .*AT&T\s+Services|
        Executive\s+Vice\s+President|
        Federal\s+Regulatory\s+Relations|
        Suite\s+\d+|
        Washington,\s*DC\s+\d{5}|
        Chairwoman\s+Jessica\s+Rosenworcel|
        Dear\s+(?:Chairwoman|Chairman|Commissioner).+:
    )
    '''
)

In [16]:
LETTER_SALUTATION_RE = re.compile(
    r'''(?ix)
    ^Dear\s+(?:Chairwoman|Chairman|Commissioner|Secretary|Ms\.|Mr\.).+:
    '''
)

In [17]:
FOOTER_LINE_RE = re.compile(
    r'''(?ix)
    ^\s*
    (?:
        (?:\#\s*){3}(?:\s+.*)?|
        \d*\s*[-\u2013\u2014]{1,2}\s*FCC\s*[-\u2013\u2014]{1,2}.*|
        FCC\s+(?:News|For)\s*$|
        Office\s+of\s+Media\s+Relations:|
        Office\s+of\s+Commissioner.+:?\s*(?:\(?202\)?|$)|
        Media\s+Relations:|
        Media\s+Contact:|
        FCC\s+Media\s+Contact:|
        ASL\s+Videophone:|
        TTY:|
        Twitter:?|
        @FCC\s*/|
        www\.fcc\.gov/(?:office-media-relations|media-relations|about/leadership|leadership)(?:/|$|\s*$)|
        This\s+is\s+an\s+unofficial\s+announcement|
        For\s+more\s+(?:news|information)
    )
    '''
)

In [18]:
INLINE_FOOTER_RE = re.compile(
    r'''(?ix)
    \s+
    (?:
        (?:\#\s*){3}|
        \d*\s*[-\u2013\u2014]{1,2}\s*FCC\s*[-\u2013\u2014]{1,2}
    )
    .*\s*$
    '''
)

## 3.2. Define Helper Functions

In [19]:
def normalize_lines(text):
    '''
    Convert raw text into clean non-empty lines.
    '''

    # Remove Byte Order Mark (\ufeff)
    # Windows- or Old Mac-style line breack -> normal line break
    # non-breaking space -> normal space
    # hyphen, non-breaking hyphen, figure dash -> normal hyphen
    text = (
        text.replace('\ufeff', '')
            .replace('\r\n', '\n')
            .replace('\r', '\n')
            .replace('\xa0', ' ')
            .replace('\u2010', '-')
            .replace('\u2011', '-')
            .replace('\u2012', '-')
    )

    # If � appears between two letters, replace it with '
    # Else, replace it with a normal space
    text = re.sub(r"(?<=[A-Za-z])\ufffd(?=[A-Za-z])", "'", text)
    text = text.replace('\ufffd', ' ')

    lines = []

    for line in text.splitlines():
        line = re.sub(r'\s+', ' ', line).strip()

        if not line:
            continue

        # Drop isolated PDF page numbers.
        if re.fullmatch(r'\d{1,3}', line):
            continue

        lines.append(line)

    return lines

In [20]:
def upper_ratio(line):
    '''
    Calculate how much of a line is uppercase.
    Useful for detecting title/header lines.
    '''
    letters = [char for char in line if char.isalpha()]

    if not letters:
        return 0

    return sum(char.isupper() for char in letters) / len(letters)

In [21]:
def has_terminal_punctuation(line):
    return bool(re.search(r'[.;?!:][\'"\)\]\u2019\u201d]*$', line.strip()))    
# \u2019: closing single quote(’)
# \u201d: closing double quote(”)

In [22]:
def title_case_ratio(line):
    words = re.findall(r"[A-Za-z][A-Za-z']+", line)

    if not words:
        return 0

    title_words = sum(word[0].isupper() for word in words)

    return title_words / len(words)

In [23]:
def is_title_like_line(line):
    '''
    Detect headline/subheadline lines before the actual body.
    '''
    if TITLE_SKIP_RE.search(line):
        return True

    if len(line) <= 220 and upper_ratio(line) >= 0.62:
        return True

    if len(line) > 220 or has_terminal_punctuation(line):
        return False

    if len(line) <= 80 and title_case_ratio(line) >= 0.72:
        return True

    return False

In [24]:
def is_front_matter_line(line):
    '''
    Detect header/title/contact lines before the actual body.
    '''
    if HEADER_RE.search(line):
        return True

    if CONTACT_RE.search(line):
        return True

    if DATE_LINE_RE.search(line):
        return True

    if LETTER_FRONT_MATTER_RE.search(line):
        return True

    if CITY_BODY_PREFIX_RE.match(line):
        first_line = CITY_BODY_PREFIX_RE.sub('', line, count=1).strip()
        if len(first_line) >= 12 and not re.match(r'(?ix)^D\.?\s*C\.?\s*\d{4,5}', first_line):
            return False

    if is_title_like_line(line):
        return True

    if len(line) < 12:
        return True

    return False

In [25]:
def is_content_line(line):
    '''
    Detect a likely first body line after front matter has been skipped.
    '''
    if FOOTER_LINE_RE.match(line):
        return False

    if is_front_matter_line(line):
        return False

    if len(line) < 8:
        return False

    if has_terminal_punctuation(line):
        return True

    if re.match(r'''^\s*(?:['"\u201c\u201d]\s*)?(?:[A-Z0-9]|\u2022)''', line):
        return True
    # \u2022: Unicode bullet character(•)

    return False

In [26]:
def trim_body_footer(body_lines):
    '''
    Remove footer/contact lines after the selected body start.
    '''
    trimmed = []

    for i, line in enumerate(body_lines):
        inline_match = INLINE_FOOTER_RE.search(line)

        if inline_match:
            before_footer = line[:inline_match.start()].strip()

            if before_footer:
                trimmed.append(before_footer)

            return trimmed, 'inline_footer_marker'

        if i > 0 and FOOTER_LINE_RE.match(line):
            if re.match(r'(?ix)^\s*(?:\#\s*){3}', line):
                method = 'footer_marker'
            elif re.match(r'(?ix)^\s*Office\s+of\s+Media\s+Relations:', line):
                method = 'office_media_footer'
            elif re.match(r'(?ix)^\s*Office\s+of\s+Commissioner', line):
                method = 'commissioner_office_footer'
            elif re.match(r'(?ix)^\s*(?:Media\s+Contact:|Media\s+Relations:)', line):
                method = 'trailing_media_contact'
            elif re.match(r'(?ix)^\s*Twitter:', line):
                method = 'twitter'
            else:
                method = 'footer_marker'

            return trimmed, method

        trimmed.append(line)

    return trimmed, 'no_footer_marker'

In [27]:
def clean_join(lines):
    '''
    Join wrapped PDF/text lines into one clean text string.
    '''
    body = ' '.join(line for line in lines if line)

    # Repair hyphenated line wraps.
    body = re.sub(r'(?<=[A-Za-z])-\s+(?=[A-Za-z])', '-', body)

    # Remove spaces before punctuation.
    body = re.sub(r'\s+([,.;:?!])', r'\1', body)

    # Collapse repeated spaces.
    body = re.sub(r'\s+', ' ', body).strip()

    return body

In [28]:
def extract_body(text):
    '''
    Extract the main body of one FCC news release.
    >>> Parameters: text (str) /Raw text from one .txt file.
    >>> Returns: body_text (str), extraction_method (str), footer_method (str)
    '''
    lines = normalize_lines(text)

    if not lines:
        return '', 'empty_text', 'no_footer_marker'

    # Method 1: Most news releases have a dateline.
    for i, line in enumerate(lines):
        if DATELINE_LINE_RE.match(line):
            first_line = DATELINE_LINE_RE.sub('', line, count=1).strip()
            body_lines = ([first_line] if first_line else []) + lines[i + 1:]
            body_lines, footer_method = trim_body_footer(body_lines)

            return clean_join(body_lines), 'dateline', footer_method

    # Method 2: Letter-style attachments should start after the salutation.
    for i, line in enumerate(lines[:80]):
        if LETTER_SALUTATION_RE.match(line):
            body_lines = lines[i + 1:]
            body_lines, footer_method = trim_body_footer(body_lines)

            return clean_join(body_lines), 'letter_salutation', footer_method

    # Method 3: Skip front matter/title/contact lines, then use the first content-like line.
    i = 0

    while i < min(80, len(lines)) and is_front_matter_line(lines[i]):
        i += 1

    for j in range(i, min(i + 12, len(lines))):
        if CITY_BODY_PREFIX_RE.match(lines[j]):
            first_line = CITY_BODY_PREFIX_RE.sub('', lines[j], count=1).strip()

            if len(first_line) >= 12 and not re.match(r'(?ix)^D\.?\s*C\.?\s*\d{4,5}', first_line):
                body_lines = ([first_line] if first_line else []) + lines[j + 1:]
                body_lines, footer_method = trim_body_footer(body_lines)

                return clean_join(body_lines), 'city_prefix_without_date', footer_method

    for j in range(i, len(lines)):
        if CITY_BODY_PREFIX_RE.match(lines[j]):
            first_line = CITY_BODY_PREFIX_RE.sub('', lines[j], count=1).strip()

            if len(first_line) >= 12 and not re.match(r'(?ix)^D\.?\s*C\.?\s*\d{4,5}', first_line):
                body_lines = ([first_line] if first_line else []) + lines[j + 1:]
                body_lines, footer_method = trim_body_footer(body_lines)

                return clean_join(body_lines), 'city_prefix_without_date', footer_method

        if is_content_line(lines[j]):
            body_lines = lines[j:]
            body_lines, footer_method = trim_body_footer(body_lines)

            return clean_join(body_lines), 'content_after_front_matter', footer_method

    # Method 4: Last fallback.
    body_lines = lines[i:]
    body_lines, footer_method = trim_body_footer(body_lines)

    return clean_join(body_lines), 'fallback_after_front_matter', footer_method

In [29]:
def read_release_text(file_path):
    file_path = Path(file_path)

    try:
        return file_path.read_text(encoding='utf-8-sig')
    except UnicodeDecodeError:
        return file_path.read_text(encoding='cp1252', errors='replace')

In [30]:
def build_body_dataframe_from_folder(text_folder):
    text_folder = Path(text_folder)

    records = []

    for file_path in sorted(text_folder.glob('*.txt')):
        
        raw_text = read_release_text(file_path)

        body_text, extraction_method, footer_method = extract_body(raw_text)

        records.append({
            'filename': file_path.name,
            'body_text': body_text,
            'body_word_count': len(body_text.split()),
            'extraction_method': extraction_method,
            'footer_method': footer_method
        })

    return pd.DataFrame(records)

## 3.3. Collect the Text

In [31]:
TEXT_FOLDER = Path('fcc_news_release_documents')

body_df = build_body_dataframe_from_folder(TEXT_FOLDER)

body_df

,filename,body_text,body_word_count,extraction_method,footer_method
0,100105_Federal_and_State_Partners_To_Test_Nati...,The Department of Homeland Security's Federal ...,558,city_prefix_without_date,footer_marker
1,100105_MEDIA_BUREAU_ANNOUNCES_PANELISTS_AND_AG...,The Media Bureau today announced the panelists...,269,city_prefix_without_date,footer_marker
2,100106_PANELISTS_ANNOUNCED_FOR_JANUARY_13_WORK...,The Federal Communications Commission will hol...,288,city_prefix_without_date,no_footer_marker
3,100107_FCC_Launches_Reboot.FCC.gov_to_Engage_P...,"Today, the Federal Communications Commission l...",327,city_prefix_without_date,footer_marker
4,"100107_Statement_of_William_T._Lake,_Chief,_Me...",Today Sinclair and Mediacom have completed the...,82,city_prefix_without_date,footer_marker
...,...,...,...,...,...
3191,260430_FCC_Modernizes_Satellite_Spectrum_Shari...,The Federal Communications Commission today vo...,262,dateline,footer_marker
3192,260430_FCC_Proposes_Strengthening_Know-Your-Cu...,In an effort to stop illegal calls before they...,303,dateline,footer_marker
3193,260430_FCC_Proposes_to_Amend_Audible_Crawl_Rul...,"Today, the Federal Communications Commission a...",292,dateline,footer_marker
3194,260430_FCC_Targets_Covered_List_Entities_Blank...,"Today, the Federal Communications Commission v...",346,dateline,footer_marker


In [32]:
print('Empty body texts:', (body_df['body_word_count'] == 0).sum())

Empty body texts: 1


In [33]:
body_df.to_csv(
    'fcc_news_release_body_texts.csv',
    index=False,
    encoding='utf-8-sig'
)

In [34]:
body_df['filename_key'] = body_df['filename'].str.lower()

metadata_df = metadata_df.merge(
    body_df,
    left_on='downloaded_filename_key',
    right_on='filename_key',
    how='left'
)

metadata_df = metadata_df.drop(columns=[
    'selected_txt_count', 'download_status', 'error_message',
    'downloaded_files', 'downloaded_filename', 'downloaded_filename_key', 'filename_key'
    ]
)

In [35]:
SPANISH_FILE_LIST = [
    '120531_Spanish_Translation,_Fact_Sheet_on_Connect2Compete_Pilot_Program.txt',
    '120807_Spanish_Translation,_PC_Pledge_100_Campaign_Fact_Sheet.txt',
    '210611_FEMA_y_FCC_planifican_prueba_del_Sistema_de_Alerta_en_Emergencias.txt',
    '230817_20_Millones_de_Hogares_Se_Inscriben_en_el_ACP.txt',
    '231019_Gomez_Provides_Remarks_In_English,_Spanish_at_October_Meeting.txt',
    '231115_Gomez_Provides_Remarks_in_English_Spanish_at_FCCs_November_Meeting.txt',
    '231213_Gomez_Provides_Remarks_in_English_Spanish_at_FCCs_December_Meeting.txt',
    '240425_Gomez_Votes_to_Restore_Net_Neutrality.txt',
    '240807_Gomez_Votes_to_Establish_New_Emergency_Alert_at_August_Meeting.txt',
]

manual_remove_file_list = [
    '210624_Commissioner_Simington_Participates_On_TIA_Broadband_Panel.txt',
    # FCC-provided original .txt file is encoding-corrupted

    '111130_Broadband_Adoption_Presentation_to_FCC_Open_Meeting.txt',
    # Empty .txt file

    '101130_FCC_IMPLEMENTATION_OF_THE_21ST_CENTURY_COMMUNICATIONS_AND_VIDEO_ACCESSIBILITY_ACT.txt'
    # Presentation slides converted to .txt file
]

NON_BODY_FILENAME_RE = re.compile(
    r'(?i)(?:broadcast_station_totals|list_of_input_channels)'
)

remove_file_mask = (
    metadata_df['filename'].isin(SPANISH_FILE_LIST + manual_remove_file_list) |
    metadata_df['filename'].str.contains(NON_BODY_FILENAME_RE, na=False)
)

for url in metadata_df.loc[remove_file_mask, 'webpage_url']:
    print(f'>>> {url}')
print(f'{sum(remove_file_mask)} documents removed')

metadata_df = metadata_df[~remove_file_mask]

>>> https://www.fcc.gov/document/gomez-votes-establish-new-emergency-alert-august-meeting
>>> https://www.fcc.gov/document/gomez-votes-restore-net-neutrality
>>> https://www.fcc.gov/document/gomez-provides-remarks-english-spanish-fccs-december-meeting
>>> https://www.fcc.gov/document/gomez-provides-remarks-english-spanish-fccs-november-meeting
>>> https://www.fcc.gov/document/gomez-provides-remarks-english-spanish-october-meeting
>>> https://www.fcc.gov/document/20-millones-de-hogares-se-inscriben-en-el-acp
>>> https://www.fcc.gov/document/broadcast-station-totals-september-30-2021
>>> https://www.fcc.gov/document/broadcast-station-totals-june-30-2021
>>> https://www.fcc.gov/document/commissioner-simington-participates-tia-broadband-panel
>>> https://www.fcc.gov/document/fema-y-fcc-planifican-prueba-del-sistema-de-alerta-en-emergencias
>>> https://www.fcc.gov/document/broadcast-station-totals-march-31-2021
>>> https://www.fcc.gov/document/broadcast-station-totals-december-31-2020
>>> h

In [36]:
metadata_df

,page_title,full_title,document_type,bureaus,description,webpage_url,selected_txt_urls,all_txt_urls,all_txt_count,released_on,issued_on,adopted,filename,body_text,body_word_count,extraction_method,footer_method
0,FCC Proposes to Amend Audible Crawl Rule to Pr...,FCC Proposes to Amend Audible Crawl Rule to Pr...,News Release,Media Relations; Space,NaN,https://www.fcc.gov/document/fcc-proposes-amen...,https://docs.fcc.gov/public/attachments/DOC-42...,https://docs.fcc.gov/public/attachments/DOC-42...,3,2026-04-30,2026-04-30,2026-04-30,260430_FCC_Proposes_to_Amend_Audible_Crawl_Rul...,"Today, the Federal Communications Commission a...",292,dateline,footer_marker
1,FCC Adopts Rules to Enhance the Integrity of t...,FCC Adopts Rules to Enhance the Integrity of t...,News Release,Media Relations; Wireline Competition,NaN,https://www.fcc.gov/document/fcc-adopts-rules-...,https://docs.fcc.gov/public/attachments/DOC-42...,https://docs.fcc.gov/public/attachments/DOC-42...,4,2026-04-30,2026-04-30,2026-04-30,260430_FCC_Adopts_Rules_to_Enhance_the_Integri...,"Today, the Federal Communications Commission a...",342,dateline,footer_marker
2,FCC Targets 'Covered List' Entities' Blanket S...,FCC Proposes Banning Entities on 'Covered List...,News Release,Media Relations; Wireline Competition,NaN,https://www.fcc.gov/document/fcc-targets-cover...,https://docs.fcc.gov/public/attachments/DOC-42...,https://docs.fcc.gov/public/attachments/DOC-42...,3,2026-04-30,2026-04-30,2026-04-30,260430_FCC_Targets_Covered_List_Entities_Blank...,"Today, the Federal Communications Commission v...",346,dateline,footer_marker
3,FCC Targets Device Test Labs in Nations Withou...,FCC Looks to Prohibit Electronic Device Testin...,News Release,Engineering & Technology; Media Relations,NaN,https://www.fcc.gov/document/fcc-targets-devic...,https://docs.fcc.gov/public/attachments/DOC-42...,https://docs.fcc.gov/public/attachments/DOC-42...,3,2026-04-30,2026-04-30,2026-04-30,260430_FCC_Targets_Device_Test_Labs_in_Nations...,"Today, the Federal Communications Commission v...",410,dateline,footer_marker
4,FCC Proposes Strengthening 'Know-Your-Customer...,FCC Proposes Strengthening 'Know-Your-Customer...,News Release,Consumer and Governmental Affairs; Media Relat...,NaN,https://www.fcc.gov/document/fcc-proposes-stre...,https://docs.fcc.gov/public/attachments/DOC-42...,https://docs.fcc.gov/public/attachments/DOC-42...,3,2026-04-30,2026-04-30,2026-04-30,260430_FCC_Proposes_Strengthening_Know-Your-Cu...,In an effort to stop illegal calls before they...,303,dateline,footer_marker
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3191,"Statement of William T. Lake, Chief, Media Bur...","Statement of William T. Lake, Chief, Media Bur...",News Release,Media,NaN,https://www.fcc.gov/document/statement-william...,https://docs.fcc.gov/public/attachments/DOC-29...,https://docs.fcc.gov/public/attachments/DOC-29...,1,2010-01-07,2010-01-07,NaT,"100107_Statement_of_William_T._Lake,_Chief,_Me...",Today Sinclair and Mediacom have completed the...,82,city_prefix_without_date,footer_marker
3192,FCC Launches Reboot.FCC.gov to Engage Public i...,FCC Launches Reboot.FCC.gov to Engage Public i...,News Release,Office of the Chairman; Media Relations,NaN,https://www.fcc.gov/document/fcc-launches-rebo...,https://docs.fcc.gov/public/attachments/DOC-29...,https://docs.fcc.gov/public/attachments/DOC-29...,1,2010-01-07,2010-01-07,NaT,100107_FCC_Launches_Reboot.FCC.gov_to_Engage_P...,"Today, the Federal Communications Commission l...",327,city_prefix_without_date,footer_marker
3193,PANELISTS ANNOUNCED FOR JANUARY 13 WORKSHOP ON...,PANELISTS ANNOUNCED FOR JANUARY 13 WORKSHOP ON...,News Release,Wireline Competition,NaN,https://www.fcc.gov/document/panelists-announc...,https://docs.fcc.gov/public/attachments/DOC-29...,https://docs.fcc.gov/public/attachments/DOC-29...,1,2010-01-06,2010-01-06,NaT,100106_PANELISTS_ANNOUNCED_FOR_JANUARY_13_WORK...,The Federal Communications Commission will hol...,288

In [37]:
print('Number of documents:', len(metadata_df))

print('\n', metadata_df['extraction_method'].value_counts())

print('\n', metadata_df['footer_method'].value_counts())

LENGTH_THRESHOLD =50
text_is_short = metadata_df['body_word_count'] < LENGTH_THRESHOLD
print(f'\n Short text (less than {LENGTH_THRESHOLD} words): {text_is_short.sum()}')

Number of documents: 3139

 extraction_method
dateline                      2188
city_prefix_without_date       804
content_after_front_matter     146
letter_salutation                1
Name: count, dtype: int64

 footer_method
footer_marker                 3025
no_footer_marker                93
commissioner_office_footer      17
trailing_media_contact           2
office_media_footer              1
inline_footer_marker             1
Name: count, dtype: int64

 Short text (less than 50 words): 9


## 3.4. Inspect the Text

In [38]:
inspect_filter = (
    metadata_df['extraction_method'].isin(['content_after_front_matter']) |
    (metadata_df['body_word_count'].fillna(0) < 50) |
    metadata_df['footer_method'].eq('no_footer_marker')
)

In [39]:
metadata_df[inspect_filter][['filename', 'body_text', 'extraction_method', 'footer_method']]

,filename,body_text,extraction_method,footer_method
101,251006_Chairman_Carr_Launches_FCCs_Space_Month...,"EL SEGUNDO, Calif., October 6, 2025—At a ribbo...",content_after_front_matter,footer_marker
160,250702_Chairman_Carr_A_Build_Agenda_for_Americ...,"SIOUX FALLS, SD, July, 2, 2025— Today, FCC Cha...",content_after_front_matter,footer_marker
176,250604_Commissioner_Geoffrey_Starks_on_His_Dep...,Federal Communications Commission Commissioner...,dateline,no_footer_marker
177,250604_Commissioner_Simington_Announces_His_De...,Federal Communications Commission Commissioner...,dateline,no_footer_marker
189,250429_Chairman_Carr_Highlights_Wins_Delivered...,"Today, Chairman Carr summarized some of the ke...",dateline,no_footer_marker
...,...,...,...,...
3185,100115_PANELISTS_ANNOUNCED_FOR_JANUARY_19_WORK...,The Federal Communications Commission will hol...,city_prefix_without_date,no_footer_marker
3186,"100115_Statement_of_P._Michele_Ellison,_Chief_...","Today, the Enforcement Bureau launches a new i...",content_after_front_matter,footer_marker
3189,100113_MEDIA_BUREAU_ANNOUNCES_PROCEDURES_FOR_O...,The Commission generally prohibits noncommerci...,content_after_front_matter,footer_marker
3190,100108_FCC_CHAIRMAN_GENACHOWSKI_STATEMENT_ON_D...,"""This case underscores the importance of the F...",content_after_front_matter,footer_marker


In [40]:
for filename in metadata_df[inspect_filter]['filename']:
    print(filename)

251006_Chairman_Carr_Launches_FCCs_Space_Month_Agenda.txt
250702_Chairman_Carr_A_Build_Agenda_for_America_Speech.txt
250604_Commissioner_Geoffrey_Starks_on_His_Departure_From_the_FCC.txt
250604_Commissioner_Simington_Announces_His_Departure.txt
250429_Chairman_Carr_Highlights_Wins_Delivered_During_First_100_Days.txt
250224_Carr_Visits_Alabama_Tower_Workers,_Public_Safety_Officials,_TV_Station.txt
250131_Chairman_Carr_in_North_Carolina_on_First_Official_Trip_as_Agency_Head.txt
250131_Commissioner_Simington_Announces_Staff_Changes.txt
250122_Starks_Announces_Black_History_Month_Event_with_Former_Chairs.txt
241118_Simington_Congratulates_Carr.txt
241108_Chairwoman_Rosenworcel_Statement_on_Offensive_Texts.txt
240807_Telecoms_Respond_to_Chairwoman_Letters_on_AI_Political_Robocall.txt
240806_Schedule_Change_August_7_Open_Meeting_Start_Time_Moved_to_115pm.txt
240409_FCC_Chairwoman_Hosts_Public_Safety_Roundtable_on_Net_Neutrality.txt
240318_Starks_Urges_Action_on_ACP.txt
240229_FCC_ICO_Strengt

In [41]:
for idx in metadata_df[inspect_filter].index:

    filename = metadata_df.loc[idx, 'filename']
    print(f'<File name {idx}: {filename}> \n')

    file_path = TEXT_FOLDER / filename
    text = read_release_text(file_path)
    print(f'[Original text] >>> {normalize_lines(text)} \n')
       
    print(f'[Text body] >>> {extract_body(text)} \n')
    print()


<File name 101: 251006_Chairman_Carr_Launches_FCCs_Space_Month_Agenda.txt> 

[Original text] >>> ['Chairman Carr Launches FCC’s ‘Space Month’ Agenda', 'Marks Major Step Forward in FCC’s Work to Promote U.S. Space Dominance', 'EL SEGUNDO, Calif., October 6, 2025—At a ribbon cutting for a new satellite manufacturing facility in The Gundo, FCC Chairman Brendan Carr announced the launch of his “Space Month” agenda. One of the core objectives of the FCC’s Build America Agenda is boosting America’s space economy and, in his remarks before industry entrepreneurs, the Chairman previewed actions that the FCC will vote on later this month to reimagine the regulatory framework for space innovation.', 'Chairman Carr issued the following statement:', '“With President Trump’s leadership, America is entering a new Golden Age of innovation in space—one where U.S. businesses are going to dominate. That is why President Trump signed an Executive Order earlier this year to streamline regulations and fost

In [42]:
comparison_output_path = Path('full_text_vs_cleaned_body.txt')

comparison_blocks = []

for idx in metadata_df.index:

    filename = metadata_df.loc[idx, 'filename']

    file_path = TEXT_FOLDER / filename
    text = read_release_text(file_path)

    original_lines = normalize_lines(text)
    original_text = '\n'.join(original_lines)
    body_text, extraction_method, footer_method = extract_body(text)

    comparison_block = f'''
============================================================
File index: {idx}
File name: {filename}
Extraction method: {extraction_method}
Footer method: {footer_method}
============================================================

[Original text]

{original_text}

------------------------------------------------------------

[Cleaned text body]

{body_text}

'''

    comparison_blocks.append(comparison_block)


comparison_output_path.write_text(
    '\n'.join(comparison_blocks),
    encoding='utf-8-sig'
)


21178983

In [43]:
metadata_df.to_parquet('fcc_news_release_metadata_with_body_text.parquet', index=False)